# Raw CSV bank involvement summary

Reads one or more IBM-style transaction CSVs from `data/raw/` and builds a wide table keyed by **bank-id**.

For each file (e.g. `HI-Medium_Trans.csv` → label `HI-Medium`):

- `txn_{label}_to_bank` — row count where this bank is **receiver** (`To Bank`).
- `txn_{label}_from_bank` — row count where this bank is **sender** (`From Bank`).
- `launder_{label}_to_bank` — same as _to_bank but only rows with laundering label true.
- `launder_{label}_from_bank` — same as _from_bank but only rows with laundering label true.

Multiple CSVs add four columns each. Large files are processed in chunks.

In [ ]:
from __future__ import annotations

import sys
from collections import Counter
from pathlib import Path

import pandas as pd

# Notebook-friendly import when package is not installed
_here = Path.cwd().resolve()
_root = _here
for _ in range(6):
    if (_root / "pyproject.toml").is_file():
        break
    _root = _root.parent
else:
    _root = _here
_src = _root / "src"
if _src.is_dir() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from aml_inspector.config import DATA_RAW
from aml_inspector.data.column_contract import (
    COL_FROM_BANK,
    COL_IS_LAUNDERING,
    COL_TO_BANK,
    validate_raw_csv,
)

DATA_RAW = DATA_RAW.resolve()
print("DATA_RAW:", DATA_RAW)

In [ ]:
# --- configuration ---
# None = all *.csv under DATA_RAW; or set explicit basenames / paths
CSV_NAMES: list[str] | None = None
CHUNKSIZE = 500_000


def discover_csv_paths(raw_dir: Path, names: list[str] | None) -> list[Path]:
    if names:
        paths: list[Path] = []
        for n in names:
            p = Path(n)
            paths.append(p if p.is_absolute() else (raw_dir / p).resolve())
        missing = [str(p) for p in paths if not p.is_file()]
        if missing:
            raise FileNotFoundError("Missing CSV(s): " + ", ".join(missing))
        return sorted(paths, key=lambda x: x.name.lower())
    found = sorted(raw_dir.glob("*.csv"), key=lambda x: x.name.lower())
    if not found:
        raise FileNotFoundError(f"No *.csv under {raw_dir}")
    return found


def txn_label_from_path(path: Path) -> str:
    """Short label for columns (strip common `_Trans` suffix from Kaggle names)."""
    stem = path.stem
    if stem.endswith("_Trans"):
        stem = stem[: -len("_Trans")]
    return stem


def unique_column_label(path: Path, used: set[str]) -> str:
    """Prefer short label; fall back to full stem / suffixed stem if names collide."""
    for cand in (txn_label_from_path(path), path.stem):
        if cand not in used:
            used.add(cand)
            return cand
    i = 2
    while True:
        cand = f"{path.stem}_{i}"
        if cand not in used:
            used.add(cand)
            return cand
        i += 1


def normalize_is_launder(series: pd.Series) -> pd.Series:
    if series.dtype == object:
        lower = series.astype(str).str.strip().str.lower()
        return lower.isin(("1", "true", "yes", "y"))
    return series.astype("Int64").fillna(0).astype(int).astype(bool)


def _bank_int_counts(series: pd.Series) -> Counter:
    s = pd.to_numeric(series, errors="coerce").dropna().astype(int)
    return Counter(s.value_counts().to_dict())


def merge_counter(a: Counter, b: Counter) -> Counter:
    out = Counter(a)
    out.update(b)
    return out


def aggregate_one_csv(path: Path, *, chunksize: int) -> dict[str, Counter]:
    validate_raw_csv(path)
    usecols = [COL_FROM_BANK, COL_TO_BANK, COL_IS_LAUNDERING]
    txn_to = Counter()
    txn_from = Counter()
    launder_to = Counter()
    launder_from = Counter()
    reader = pd.read_csv(path, chunksize=chunksize, usecols=usecols, low_memory=False)
    for chunk in reader:
        y = normalize_is_launder(chunk[COL_IS_LAUNDERING])
        txn_to = merge_counter(txn_to, _bank_int_counts(chunk[COL_TO_BANK]))
        txn_from = merge_counter(txn_from, _bank_int_counts(chunk[COL_FROM_BANK]))
        launder_to = merge_counter(launder_to, _bank_int_counts(chunk.loc[y, COL_TO_BANK]))
        launder_from = merge_counter(
            launder_from, _bank_int_counts(chunk.loc[y, COL_FROM_BANK])
        )
    return {
        "txn_to": txn_to,
        "txn_from": txn_from,
        "launder_to": launder_to,
        "launder_from": launder_from,
    }


def counter_to_column(bank_ids: list[int], ctr: Counter, *, dtype: str = "int64") -> pd.Series:
    return pd.Series({b: int(ctr.get(b, 0)) for b in bank_ids}, dtype=dtype)


def build_summary_frame(
    path_label_pairs: list[tuple[Path, str]], *, chunksize: int
) -> pd.DataFrame:
    per_file: list[tuple[str, dict[str, Counter]]] = []
    all_banks: set[int] = set()
    for path, label in path_label_pairs:
        ag = aggregate_one_csv(path, chunksize=chunksize)
        per_file.append((label, ag))
        for c in ag.values():
            all_banks.update(c.keys())
    banks = sorted(all_banks)
    out = pd.DataFrame({"bank-id": banks})
    for label, ag in per_file:
        out[f"txn_{label}_to_bank"] = counter_to_column(banks, ag["txn_to"]).values
        out[f"txn_{label}_from_bank"] = counter_to_column(banks, ag["txn_from"]).values
        out[f"launder_{label}_to_bank"] = counter_to_column(banks, ag["launder_to"]).values
        out[f"launder_{label}_from_bank"] = counter_to_column(
            banks, ag["launder_from"]
        ).values
    return out


csv_paths = discover_csv_paths(DATA_RAW, CSV_NAMES)
_used: set[str] = set()
path_labels = [(p, unique_column_label(p, _used)) for p in csv_paths]
print("Files (column label suffix):")
for p, lab in path_labels:
    print(f"  {p.name} -> {lab}")

summary_df = build_summary_frame(path_labels, chunksize=CHUNKSIZE)
summary_df

In [ ]:
# Optional: write summary next to raw data
# out_path = DATA_RAW.parent / "interim" / "raw_bank_summary.parquet"
# out_path.parent.mkdir(parents=True, exist_ok=True)
# summary_df.to_parquet(out_path, index=False)
# print("Wrote", out_path)